In [ ]:
# Full Data Science Pipeline - Salary Prediction

# =========================================
# 1. IMPORT LIBRARIES
# =========================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor

# =========================================
# 2. LOAD DATASETS
# =========================================
df1 = pd.read_csv('Web-Developers-Salaries-Egypt-2024.csv')
df2 = pd.read_csv('wuzzuf_jobs.csv')

print("DF1 Shape:", df1.shape)
print("DF2 Shape:", df2.shape)

# =========================================
# 3. DATA INTEGRATION
# =========================================
df1.columns = df1.columns.str.lower()
df2.columns = df2.columns.str.lower()

df = pd.concat([df1, df2], ignore_index=True)

print("Combined Shape:", df.shape)

# =========================================
# 4. DATA CLEANING
# =========================================
df = df.dropna(subset=['salary'])

df['salary'] = df['salary'].astype(str)
df['salary'] = df['salary'].str.replace(',', '')
df['salary'] = df['salary'].str.extract('(\d+)')
df['salary'] = df['salary'].astype(float)

df = df.dropna()

# =========================================
# 5. ENCODING
# =========================================
le = LabelEncoder()
for col in df.select_dtypes(include='object').columns:
    df[col] = le.fit_transform(df[col])

# =========================================
# 6. SPLIT
# =========================================
X = df.drop('salary', axis=1)
y = df['salary']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# =========================================
# 7. SCALING
# =========================================
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# =========================================
# 8. VISUALIZATION
# =========================================
plt.figure()
sns.histplot(df['salary'], kde=True)
plt.title("Salary Distribution")
plt.show()

plt.figure()
sns.heatmap(df.corr(), annot=False)
plt.title("Correlation Heatmap")
plt.show()

# =========================================
# 9. MODELING + EVALUATION
# =========================================
models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(),
    "Random Forest": RandomForestRegressor()
}

results = []

for name, model in models.items():

    if name == "Linear Regression":
        model.fit(X_train_scaled, y_train)
        preds = model.predict(X_test_scaled)
    else:
        model.fit(X_train, y_train)
        preds = model.predict(X_test)

    mae = mean_absolute_error(y_test, preds)
    mse = mean_squared_error(y_test, preds)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, preds)

    results.append([name, mae, rmse, r2])

results_df = pd.DataFrame(results, columns=["Model", "MAE", "RMSE", "R2"])
print(results_df)

best_model = results_df.sort_values(by='R2', ascending=False)
print("\nBest Model:")
print(best_model.head(1))
